# ExLightNet supplementary Kaggle runner

CUDA execution: NOT RUN locally. This thin notebook installs uploaded source and calls the package CLI. Follow docs/KAGGLE_RUNBOOK.md; upload prepared arrays, not the raw CSV.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SOURCE = Path("/kaggle/input/exlightnet-code")
PREPARED = Path("/kaggle/input/exlightnet-prepared")
SITE = Path("/kaggle/working/exlightnet_python")
assert (SOURCE / "pyproject.toml").is_file(), "Set SOURCE to the uploaded code folder"
assert (PREPARED / "manifest.json").is_file(), "Set PREPARED to the exported array folder"
assert sys.version_info >= (3, 11), "Python >= 3.11 is required"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "--upgrade", "--target", str(SITE), str(SOURCE)], check=True)
os.environ["PYTHONPATH"] = str(SITE)
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Source installed offline; prepared files stay read-only inputs.")

## Select one study

Run probe first. Then use B1, B3, B2, B4, B5 in that order. The probe prints measured epoch cost and approximate projections before long studies. A new Save & Run All session needs a surviving probe/results bundle restored.

Dated amendment 2026-11-18: use ONLY=neural with CUDA after the probe, and ONLY=trees with DEVICE=cpu in a separate session of the same gpu_v2 campaign. Trees need no neural probe. B1/B2/B3 are seed-major; incomplete seed blocks are explicitly excluded from paired inference until shards are merged.

In [ ]:
STUDY = "probe"
DEVICE = "cuda"
ONLY = "all"  # all, neural, trees; trees may use DEVICE=cpu without a neural probe
CAMPAIGN = "gpu_v2"
TIME_BUDGET_HOURS = 8
RESUME_ZIP = None  # Example: /kaggle/input/exlightnet-resume/gpu_v2.zip
command = [sys.executable, str(SOURCE / "scripts/kaggle_campaign.py"),
    "--prepared-dir", str(PREPARED), "--output", "/kaggle/working",
    "--study", STUDY, "--device", DEVICE, "--only", ONLY, "--campaign", CAMPAIGN,
    "--time-budget-hours", str(TIME_BUDGET_HOURS), "--zip-every", "5"]
if RESUME_ZIP:
    command += ["--resume-zip", str(RESUME_ZIP)]
subprocess.run(command, check=True, env=os.environ.copy())

## Preserve and download outputs

Download /kaggle/working/gpu_v2.zip regularly and after completion. Save notebook outputs as documented in the runbook. Restore the downloaded bundle in a new session; completed fingerprints are skipped. Aggregate downloaded ZIPs locally with scripts/aggregate_gpu_v2.py. This notebook contains no training or scoring implementation.